# 1. Project Configuration

In [171]:
import sys

print(sys.version)
print(sys.executable)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
e:\ZaWolf_project\.venv\Scripts\python.exe


# 2. Project Paths

In [172]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
INPUT_DIR = DATA_DIR / "input"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = DATA_DIR / "output"
MODELS_DIR = PROJECT_ROOT / "models"
LOGS_DIR = PROJECT_ROOT / "logs"

for directory in [
    INPUT_DIR,
    PROCESSED_DIR,
    OUTPUT_DIR,
    MODELS_DIR,
    LOGS_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

# 3. Supported File Types

In [173]:
SUPPORTED_EXTENSIONS = {
    ".pdf": "pdf",
    ".jpg": "image",
    ".jpeg": "image",
    ".png": "image",
    ".webp": "image",
    ".xlsx": "excel",
    ".xls": "excel",
    ".csv": "csv",
    ".docx": "word",
    ".doc": "word"
}

# 3. File Type Detection

In [174]:
def detect_file_type(file_path):
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    extension = file_path.suffix.lower()
    file_type = SUPPORTED_EXTENSIONS.get(extension)

    return {
        "file_name": file_path.name,
        "file_path": str(file_path),
        "extension": extension,
        "file_type": file_type or "unsupported",
        "supported": file_type is not None
    }

# 4. Input File Loader

In [175]:
def get_input_files():
    return [
        file_path
        for file_path in INPUT_DIR.iterdir()
        if file_path.is_file()
    ]

# 5. Input File Detection

In [176]:
input_files = get_input_files()

detected_files = [
    detect_file_type(file_path)
    for file_path in input_files
]

# 6. File Routing Logic

In [177]:
def route_file(file_info):
    file_type = file_info["file_type"]

    routes = {
        "pdf": "pdf_pipeline",
        "image": "image_pipeline",
        "excel": "excel_pipeline",
        "csv": "csv_pipeline",
        "word": "word_pipeline"
    }

    return routes.get(file_type, "unsupported")

# 7. Apply File Routing

In [178]:
for file_info in detected_files:
    file_info["route"] = route_file(file_info)

# 9. PDF Type Detection

In [179]:
import fitz

def detect_pdf_type(file_path):
    file_path = Path(file_path)

    if file_path.suffix.lower() != ".pdf":
        raise ValueError("The provided file is not a PDF file.")

    document = fitz.open(file_path)

    text_length = sum(
        len(page.get_text("text").strip())
        for page in document
    )

    document.close()

    pdf_type = "digital" if text_length > 0 else "scanned"

    return {
        "file_name": file_path.name,
        "file_path": str(file_path),
        "pdf_type": pdf_type,
        "text_length": text_length
    }

## 9.2 PDF Processing

In [180]:
def process_pdf(file_info):
    pdf_type_info = detect_pdf_type(file_info["file_path"])
    file_info.update(pdf_type_info)
    return file_info

## 9.3 Process PDF Inputs

In [181]:
pdf_files = [
    file_info
    for file_info in detected_files
    if file_info["file_type"] == "pdf"
]

pdf_files = [
    process_pdf(file_info)
    for file_info in pdf_files
]

# 10. Digital PDF Text Extraction

In [182]:
def extract_pdf_text(file_path):
    file_path = Path(file_path)

    if file_path.suffix.lower() != ".pdf":
        raise ValueError("The provided file is not a PDF file.")

    document = fitz.open(file_path)

    pages = [
        page.get_text("text").strip()
        for page in document
    ]

    document.close()

    text = "\n".join(
        page_text
        for page_text in pages
        if page_text
    )

    return {
        "file_name": file_path.name,
        "file_path": str(file_path),
        "text": text,
        "text_length": len(text)
    }

## 10.1 Memory Management

In [183]:
import psutil

def get_memory_info():
    memory = psutil.virtual_memory()

    return {
        "total_gb": round(memory.total / (1024 ** 3), 2),
        "available_gb": round(memory.available / (1024 ** 3), 2),
        "used_percent": memory.percent
    }

## 10.2 Dynamic Batch Sizing

In [184]:
def calculate_batch_size():
    memory = psutil.virtual_memory()

    available_bytes = memory.available

    safety_ratio = 0.50
    memory_budget = available_bytes * safety_ratio

    estimated_page_memory = 1 * 1024 * 1024

    batch_size = max(
        1,
        int(memory_budget / estimated_page_memory)
    )

    return {
        "available_memory_gb": round(
            available_bytes / (1024 ** 3), 2
        ),
        "memory_budget_gb": round(
            memory_budget / (1024 ** 3), 2
        ),
        "batch_size": batch_size
    }

## 10.3 Dynamic Page Batching

In [185]:
MAX_BATCH_TEXT_SIZE = 12000

def get_dynamic_batch_limit():
    memory = psutil.virtual_memory()

    available_bytes = memory.available
    safety_ratio = 0.25
    memory_budget = int(available_bytes * safety_ratio)

    return {
        "available_bytes": available_bytes,
        "memory_budget_bytes": memory_budget
    }

## 10.4 Page-by-Page PDF Extraction

In [186]:
def extract_pdf_pages(file_path):
    file_path = Path(file_path)

    if file_path.suffix.lower() != ".pdf":
        raise ValueError("The provided file is not a PDF file.")

    document = fitz.open(file_path)

    try:
        batch = []
        batch_size_bytes = 0
        batch_text_size = 0

        for page_number, page in enumerate(document, start=1):
            page_text = page.get_text("text").strip()

            if not page_text:
                continue

            page_size_bytes = len(page_text.encode("utf-8"))
            page_text_size = len(page_text)

            memory_limit = get_dynamic_batch_limit()
            memory_budget = memory_limit["memory_budget_bytes"]

            exceeds_text_limit = (
                batch_text_size + page_text_size > MAX_BATCH_TEXT_SIZE
            )

            exceeds_memory_limit = (
                batch_size_bytes + page_size_bytes > memory_budget
            )

            if batch and (exceeds_text_limit or exceeds_memory_limit):
                yield batch

                batch = []
                batch_size_bytes = 0
                batch_text_size = 0

            batch.append({
                "page_number": page_number,
                "text": page_text,
                "text_length": page_text_size
            })

            batch_size_bytes += page_size_bytes
            batch_text_size += page_text_size

        if batch:
            yield batch

    finally:
        document.close()

## 10.5 Text Chunking and Overlap

In [187]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [188]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=300,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

In [189]:
def create_text_chunks(page_batch):
    chunks = []

    for page_data in page_batch:
        page_text = page_data["text"]

        page_chunks = text_splitter.split_text(page_text)

        for chunk_index, chunk_text in enumerate(page_chunks, start=1):
            chunks.append({
                "page_number": page_data["page_number"],
                "chunk_index": chunk_index,
                "text": chunk_text,
                "text_length": len(chunk_text)
            })

    return chunks

## 10.6 PDF Batch to Text Chunks

In [190]:
def process_pdf_to_chunks(file_path):
    for page_batch in extract_pdf_pages(file_path):
        yield create_text_chunks(page_batch)

In [191]:
digital_pdf_chunks = []

for file_info in pdf_files:
    if file_info["pdf_type"] == "digital":
        for chunk_batch in process_pdf_to_chunks(file_info["file_path"]):
            digital_pdf_chunks.extend(chunk_batch)

## 10.7 Context-Preserving Chunking

In [192]:
def create_contextual_chunks(page_batch, file_name):
    chunks = []

    page_texts = [
        page_data["text"]
        for page_data in page_batch
        if page_data["text"]
    ]

    if not page_texts:
        return chunks

    combined_text = "\n\n".join(page_texts)

    page_chunks = text_splitter.split_text(combined_text)

    for chunk_index, chunk_text in enumerate(page_chunks, start=1):
        chunks.append({
            "file_name": file_name,
            "chunk_index": chunk_index,
            "text": chunk_text,
            "text_length": len(chunk_text)
        })

    return chunks

### 10.7.1 Test Contextual Chunking

for file_info in pdf_files:
    if file_info["pdf_type"] != "digital":
        continue

    print(f"File: {file_info['file_name']}")

    total_batches = 0
    total_pages = 0
    total_chunks = 0

    for batch_number, page_batch in enumerate(
        extract_pdf_pages(file_info["file_path"]),
        start=1
    ):
        chunks = create_contextual_chunks(
            page_batch,
            file_info["file_name"]
        )

        total_batches += 1
        total_pages += len(page_batch)
        total_chunks += len(chunks)

        print(
            f"Batch {batch_number}: "
            f"{len(page_batch)} pages → "
            f"{len(chunks)} chunks"
        )

    print("\nSummary:")
    print(f"Total batches: {total_batches}")
    print(f"Total pages: {total_pages}")
    print(f"Total chunks: {total_chunks}")

## 11.1 Word Text Extraction

In [193]:
from docx import Document

In [194]:
def extract_word_content(file_path):
    file_path = Path(file_path)

    if file_path.suffix.lower() != ".docx":
        raise ValueError("The provided file is not a DOCX file.")

    document = Document(file_path)

    content = []

    for paragraph_number, paragraph in enumerate(
        document.paragraphs,
        start=1
    ):
        text = paragraph.text.strip()

        if text:
            content.append({
                "source_type": "paragraph",
                "source_number": paragraph_number,
                "text": text,
                "text_length": len(text)
            })

    for table_number, table in enumerate(
        document.tables,
        start=1
    ):
        rows = []

        for row in table.rows:
            cells = [
                cell.text.strip()
                for cell in row.cells
            ]

            if any(cells):
                rows.append(cells)

        if rows:
            table_text = "\n".join(
                " | ".join(row)
                for row in rows
            )

            content.append({
                "source_type": "table",
                "source_number": table_number,
                "text": table_text,
                "text_length": len(table_text)
            })

    return content

## 11.2 Dynamic Text Batching

In [195]:
def batch_text_units(text_units):
    batch = []
    batch_size_bytes = 0
    batch_text_size = 0

    for unit in text_units:
        text = unit["text"]

        unit_size_bytes = len(text.encode("utf-8"))
        unit_text_size = len(text)

        memory_limit = get_dynamic_batch_limit()
        memory_budget = memory_limit["memory_budget_bytes"]

        exceeds_text_limit = (
            batch_text_size + unit_text_size > MAX_BATCH_TEXT_SIZE
        )

        exceeds_memory_limit = (
            batch_size_bytes + unit_size_bytes > memory_budget
        )

        if batch and (exceeds_text_limit or exceeds_memory_limit):
            yield batch

            batch = []
            batch_size_bytes = 0
            batch_text_size = 0

        batch.append(unit)

        batch_size_bytes += unit_size_bytes
        batch_text_size += unit_text_size

    if batch:
        yield batch

## 11.3 Text Chunking + Overlap

In [196]:
def create_text_chunks(text_batch, file_name):
    chunks = []

    combined_text = "\n\n".join(
        unit["text"]
        for unit in text_batch
        if unit["text"]
    )

    if not combined_text:
        return chunks

    text_chunks = text_splitter.split_text(combined_text)

    for chunk_index, chunk_text in enumerate(
        text_chunks,
        start=1
    ):
        chunks.append({
            "file_name": file_name,
            "chunk_index": chunk_index,
            "text": chunk_text,
            "text_length": len(chunk_text)
        })

    return chunks

## 11.4 Unified Text Processing

In [197]:
def process_text_units(text_units, file_name):
    for text_batch in batch_text_units(text_units):
        yield create_text_chunks(
            text_batch,
            file_name
        )

## 11.5 Word Processing Pipeline

In [198]:
def process_word(file_path):
    file_path = Path(file_path)

    text_units = extract_word_content(file_path)

    yield from process_text_units(
        text_units,
        file_path.name
    )

## 11.6 Word Pipeline Testing

word_files = [
    file_info
    for file_info in detected_files
    if file_info["file_type"] == "word"
]

for file_info in word_files:
    print(f"File: {file_info['file_name']}")

    total_batches = 0
    total_chunks = 0

    for batch_number, chunks in enumerate(
        process_word(file_info["file_path"]),
        start=1
    ):
        total_batches += 1
        total_chunks += len(chunks)

        print(
            f"Batch {batch_number}: "
            f"{len(chunks)} chunks"
        )

    print("\nSummary:")
    print(f"Total batches: {total_batches}")
    print(f"Total chunks: {total_chunks}")
    print("-" * 60)

# 12. Excel / CSV Processing

## 12.1 Direct Data Extraction

In [199]:
import pandas as pd

In [213]:
def extract_structured_data(file_info):
    file_path = Path(file_info["file_path"])
    file_type = file_info["file_type"]

    if file_type == "csv":
        sheets = {
            "default": pd.read_csv(file_path)
        }

    elif file_type == "excel":
        sheets = pd.read_excel(
            file_path,
            sheet_name=None
        )

    else:
        raise ValueError(
            "The provided file is not an Excel or CSV file."
        )

    return {
        "file_name": file_path.name,
        "file_path": str(file_path),
        "file_type": file_type,
        "sheets": sheets
    }

## 12.2 Structured Data Output

In [215]:
for file_info in structured_files:
    result = extract_structured_data(file_info)

    print(f"File: {result['file_name']}")
    print(f"Type: {result['file_type']}")

    for sheet_name, data in result["sheets"].items():
        print(
            f"Sheet: {sheet_name} | "
            f"Rows: {len(data)} | "
            f"Columns: {len(data.columns)}"
        )

        print(
            f"Columns: {data.columns.tolist()}"
        )

        display(data)

File: Covid-19 Dataset .csv
Type: csv
Sheet: default | Rows: 3000 | Columns: 27
Columns: ['Patient_ID', 'Age', 'Gender', 'Region', 'Preexisting_Condition', 'Date_of_Infection', 'COVID_Strain', 'Symptoms', 'Severity', 'Hospitalized', 'Hospital_Admission_Date', 'Hospital_Discharge_Date', 'ICU_Admission', 'Ventilator_Support', 'Recovered', 'Date_of_Recovery', 'Reinfection', 'Date_of_Reinfection', 'Vaccination_Status', 'Vaccine_Type', 'Doses_Received', 'Date_of_Last_Dose', 'Long_COVID_Symptoms', 'Occupation', 'Smoking_Status', 'BMI', 'Recovery_Classification']


,Patient_ID,Age,Gender,Region,Preexisting_Condition,Date_of_Infection,COVID_Strain,Symptoms,Severity,Hospitalized,...,Date_of_Reinfection,Vaccination_Status,Vaccine_Type,Doses_Received,Date_of_Last_Dose,Long_COVID_Symptoms,Occupation,Smoking_Status,BMI,Recovery_Classification
0,1,69,Male,Hovedstaden,Obesity,2022-06-21,Delta,Mild,Moderate,Yes,...,2022-12-15,Yes,NaN,1,2022-09-22,NaN,Healthcare,Never,27.7,Delayed Recovery
1,2,38,Male,Sjælland,Asthma,2024-02-02,XBB.1.5,Mild,Moderate,No,...,2024-06-08,No,NaN,0,2023-08-21,NaN,Healthcare,Never,21.9,Typical Recovery
2,3,41,Female,Syddanmark,Hypertension,2023-05-28,Beta,Mild,High,Yes,...,2023-12-19,Yes,Janssen,3,2024-05-14,NaN,Unemployed,Never,22.7,Delayed Recovery
3,4,81,Female,Hovedstaden,Asthma,2023-08-13,Delta,Severe,High,No,...,2024-08-24,Yes,AstraZeneca,1,2024-10-31,NaN,Office Worker,Never,27.7,Delayed Recovery
4,5,50,Female,Syddanmark,Cardiovascular,2023-03-10,Delta,Mild,High,No,...,2023-09-08,Yes,NaN,2,2023-07-05,NaN,Student,Never,11.9,Delayed Recovery
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,2996,43,Male,Nordjylland,Hypertension,2022-10-19,XBB.1.5,Severe,Critical,No,...,2023-04-03,Yes,NaN,1,2024-09-20,NaN,Driver,Never,22.0,Delayed Recovery
2996,2997,36,Female,Syddanmark,Obesity,2022-12-16,Omicron,Moderate,Low,No,...,2023-07-28,Yes,Pfizer,2,2023-10-05,NaN,Healthcare,Never,27.8,NaN
2997,2998,75,Female,Sjælland,Cardiovascular,2023-09-30,Beta,Severe,Moderate,No,...,2024-03-15,Yes,Moderna,3,2023-05-13,NaN,Teacher,Former,20.9,Typical Recovery
2998,2999,45,Female,Hovedstaden,Asthma,2023-06-06,Delta,Severe,Moderate,No,...,2023-09-26,Yes,AstraZeneca,1,2024-05-13,NaN,Student,Never,19.3,Typical Recovery
